In [0]:
import pandas as pd
from pyspark.sql import functions as F, Window

BASE = "/Volumes/workspace/equipment/raw"
COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

def parse(tbl):
    p = F.split(F.trim("value"), r"\s+")
    return (spark.table(tbl)
        .select(*[(p[i].cast("int") if c in ("unit", "cycle") else p[i].cast("double")).alias(c)
                  for i, c in enumerate(COLS)])
        .filter(F.col("unit").isNotNull() & F.col("cycle").isNotNull())
        .dropDuplicates(["unit", "cycle"]))

wmax = Window.partitionBy("unit")

train = (parse("workspace.equipment.bronze_train")
    .withColumn("max_cycle", F.max("cycle").over(wmax))
    .withColumn("rul", F.col("max_cycle") - F.col("cycle")))
train.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.equipment.silver_train")

# test files stop before failure; the true final RUL per engine comes from RUL_FD001.txt
final = pd.read_csv(f"{BASE}/rul/RUL_FD001.txt", header=None, names=["final_rul"])
final["unit"] = range(1, len(final) + 1)
final_df = spark.createDataFrame(final)

test = (parse("workspace.equipment.bronze_test")
    .withColumn("max_cycle", F.max("cycle").over(wmax))
    .join(final_df, "unit")
    .withColumn("rul", F.col("final_rul") + F.col("max_cycle") - F.col("cycle")))
test.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.equipment.silver_test")

print("train:", train.count(), "| test:", test.count(), "| engines:", train.select("unit").distinct().count())